# Deep Research
One of the classic cross-business Agentic use cases! This is huge.

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [2]:
load_dotenv(override=True)

True

In [3]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

# Strategy for the Deep Research Agent
We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

# We will build 4 Agents:
1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.

# Agent 1: The Search Agent

## OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

- `WebSearchTool` lets an agent search the web.
- `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.
- `CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.
- `HostedMCPTool` exposes a remote MCP server's tools to the model.
- `ImageGenerationTool` generates images from a prompt.
- `ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.

## Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing

In [5]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [6]:
# Initialize the search agent with the specified instructions, tools, model, and model settings
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [7]:
# run agent to perform search task
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

In 2026, the most commonly cited and widely used AI agent frameworks are LangChain, LangGraph, CrewAI, AutoGen, Semantic Kernel, Haystack, Agno, Swarm, Atomic Agents, and Pydantic AI. Several 2026 curated rankings and comparison lists consistently place LangChain at or near the top for overall popularity, with LangGraph, CrewAI, and AutoGen also ranking highly for agent orchestration and multi-agent workflows. ([github.com](https://github.com/vishwasvijayabaskar-code/awesome-ai-agents?utm_source=openai))

If you’re looking for the “best known” options by use case: LangGraph is often favored for stateful, graph-based workflows; CrewAI for role-based multi-agent teams; AutoGen for conversational multi-agent systems; Semantic Kernel for enterprise app integration; and Pydantic AI for type-safe, structured agents. Recent comparison guides and momentum trackers also highlight newer contenders like OpenAI Agents SDK, Google ADK, Claude Agent SDK, and Mastra as rising frameworks in 2026. ([github.com](https://github.com/langfuse/langfuse-docs/blob/main/content/blog/2025-03-19-ai-agent-comparison.mdx?utm_source=openai))

## As always, take a look at the trace
https://platform.openai.com/traces

# Agent 2: The Planner Agent
### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# Define the structure for web search items and plans
# this class defines the structure for individual web search items and a collection of them in a search plan.
# pydantic BaseModel is used to define the structure of the data.
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

# pydantic BaseModel is used to define the structure of the data.
# this class defines the structure for a collection of web search items in a search plan.
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [10]:
# Display the JSON schema for the WebSearchPlan model
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [ ]:
# define the instructions for the planner agent
INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

# initialize the planner agent with the specified instructions, model, and output type
planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [ ]:
# run the planner agent to generate a web search plan for the given task
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find current market leaders and adoption data for AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 adoption market share'), WebSearchItem(reason='Identify recent rankings, surveys, or benchmark reports comparing agent frameworks.', query='2026 AI agent framework comparison survey benchmark'), WebSearchItem(reason='Check community popularity signals like GitHub stars, activity, and package downloads.', query='AI agent frameworks GitHub stars downloads 2026'), WebSearchItem(reason='Find industry articles and expert lists naming the top frameworks in 2026.', query='top AI agent frameworks 2026 article list'), WebSearchItem(reason='Look for developer discussions and ecosystem references about widely used frameworks.', query='best AI agent frameworks 2026 LangChain CrewAI AutoGen LangGraph LlamaIndex')])

# Agent 3: The Writer Agent

In [13]:
# Define the instructions and structure for the writer agent
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""

# Define the structure for the report data that the writer agent will output
# pydantic BaseModel is used to define the structure of the data.
# this class defines the structure for the report data that the writer agent will output.
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

# initialize the writer agent with the specified instructions, model, and output type
writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

In [14]:
# Define a tool to send emails to sales prospects
# This tool allows sending emails to all sales prospects with a specified subject and body.
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [15]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [17]:
# Define the instructions and initialize the email agent
# The email agent will use the send_email_tool to send emails based on the provided report.
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

# Initialize the email agent with the specified instructions, tools, and model
# The email agent will be responsible for sending emails using the send_email_tool.
email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

# Now to Orchestrate by Code
The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [ ]:
# Define functions to run web searches based on a query and handle individual search items.
# The run_searches function plans and executes multiple searches concurrently.
# The search function handles a single search item using the search agent.
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

# Perform a single web search for a given item using the search agent.
# Args:
#     item: The WebSearchItem representing the search query and reason.
# Returns:
#     The final output of the search agent for the given item.
async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [19]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

Showtime!

In [20]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


# As always, take a look at the trace
https://platform.openai.com/traces